# Tráfico en Madrid: Análisis Exploratorio de Datos

Este es el primer notebook del proyecto y su función es la más importante de todas: entender los datos antes de construir nada encima. Antes de montar un modelo de machine learning hay que saber qué aspecto tienen los datos, si hay fallos de medición, qué variables aportan información y cuáles no, y qué decisiones de limpieza vamos a tener que tomar después. Todo lo que venga en los notebooks siguientes se apoyará en las conclusiones que saquemos aquí.

El objetivo del proyecto es predecir si una vía de Madrid va a estar **congestionada** en función de la hora, el día de la semana y el sensor. La variable que mide la congestión se llama `carga` y va de 0 a 100%. El umbral a partir del cual consideramos que hay atasco lo decidiremos en este mismo notebook, basándonos en cómo se distribuye la variable.

El plan del notebook es el siguiente. Primero descargamos los datos directamente del portal del Ayuntamiento. Luego los cargamos y les echamos un vistazo general. Después analizamos en detalle la variable objetivo y los fallos de sensor, que son el ruido principal del dataset. A continuación estudiamos los patrones temporales (por hora, día y mes) y espaciales (por distrito y coordenadas). Terminamos mirando las correlaciones entre variables y resumiendo las ideas clave que van a guiar los notebooks siguientes.

## Preparación del entorno

En la primera celda de código dejamos todos los imports, las rutas y la configuración estética. Hacerlo al inicio tiene dos ventajas: por un lado, cualquier persona que abra el notebook ve de un golpe qué librerías necesita, y por otro, cualquier cambio de configuración (por ejemplo, el tamaño por defecto de las figuras) se aplica en un único sitio y afecta al resto del notebook.

In [ ]:
# Imports de la librería estándar
import zipfile
from pathlib import Path

# Imports de terceros
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


`requests` lo usamos para descargar los ZIP directamente del portal de datos. `zipfile` nos permite leer el CSV que hay dentro de cada ZIP sin tener que extraerlo a disco primero, lo que nos ahorra duplicar ficheros grandes. El resto son las librerías habituales del stack de data science en Python.

Seguidamente definimos las rutas del proyecto. Trabajar con `pathlib.Path` en lugar de concatenar strings evita problemas de barras `/` vs `\` en Windows y deja el código mucho más legible.

In [ ]:
# Rutas de datos. DATA_RAW contiene los ficheros crudos descargados tal cual del portal.
DATA_RAW  = Path('../data/raw')
DATA_HIST = DATA_RAW / 'historico'     # ZIPs mensuales de mediciones
DATA_SENS = DATA_RAW / 'sensores'      # CSV con la metadata de cada sensor

# Nos aseguramos de que las carpetas existen antes de intentar escribir en ellas
DATA_HIST.mkdir(parents=True, exist_ok=True)
DATA_SENS.mkdir(parents=True, exist_ok=True)


Las constantes globales del proyecto van en mayúsculas y se definen también aquí. `RANDOM_STATE` se usará en cualquier operación que tenga aleatoriedad (muestreos, splits, modelos) para que los resultados sean reproducibles. Si alguien reejecuta el notebook debería obtener exactamente los mismos números que nosotros.

In [ ]:
RANDOM_STATE = 42


La configuración de pandas controla cómo se muestran los DataFrames al imprimirlos. Sin estas dos líneas, pandas trunca columnas y muestra números con siete decimales, lo que dificulta leer las tablas.

In [ ]:
pd.set_option('display.max_columns', None)         # Ver todas las columnas al hacer .head()
pd.set_option('display.float_format', '{:.2f}'.format)  # Dos decimales por defecto


El tema visual. Este bloque es más largo pero merece la pena porque todas las gráficas del notebook van a heredar esta configuración y nos ahorra tener que repetir parámetros una y otra vez. Apuntamos a un estilo sobrio: ejes limpios sin recuadro arriba ni a la derecha, grid punteado muy suave, tipografía pequeña, títulos alineados a la izquierda. Es la estética que usan publicaciones como The Economist o el FT, y queda mucho más presentable que los defaults de matplotlib.

In [ ]:
sns.set_theme(style='ticks', palette='muted', font_scale=0.9)

plt.rcParams.update({
    'figure.figsize':    (10, 4),     # Ancha y corta: cabe bien en PDF a una página
    'figure.dpi':        100,         # Resolución razonable sin inflar el fichero
    'axes.spines.top':   False,       # Quitamos los bordes superior y derecho
    'axes.spines.right': False,
    'axes.titleweight':   'normal',
    'axes.titlelocation': 'left',     # Títulos alineados a la izquierda, más limpio
    'axes.titlesize':  11,
    'axes.labelsize':  10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.frameon':  False,         # Leyenda sin recuadro
    'axes.grid':       True,
    'grid.linestyle':  ':',           # Grid con líneas punteadas
    'grid.alpha':      0.35,          # Muy suave, no distrae
})

# Paleta acotada de colores que usaremos a lo largo del notebook
COLOR_PRIMARIO   = '#4C72B0'  # Azul � lo usamos para la clase "fluido" o el flujo principal
COLOR_SECUNDARIO = '#DD8452'  # Naranja � lo usamos para "congestionado", fin de semana o errores


## 1. Fuente de los datos

Los datos provienen del [portal de datos abiertos del Ayuntamiento de Madrid](https://datos.madrid.es). Trabajamos con dos datasets complementarios.

El primero es el **histórico de mediciones**. El Ayuntamiento publica un fichero ZIP por mes que contiene un CSV con todas las lecturas del mes. Cada fila del CSV es la medición de un sensor en un instante concreto. Las columnas que nos importan son `id` (identificador del sensor), `fecha` (timestamp de la lectura), `intensidad` (vehículos por hora que pasan por el punto), `ocupacion` (porcentaje del tiempo que el sensor tuvo un vehículo encima) y `carga` (índice de saturación de 0 a 100, calculado por el propio Ayuntamiento combinando intensidad y ocupación). Hay también una columna `error` que marca explícitamente las lecturas defectuosas.

El segundo dataset es la **metadata de los sensores**: un CSV donde cada fila es un sensor con su nombre de calle, distrito, tipo de elemento y coordenadas UTM. Se une al histórico por la columna `id`.

Los sensores graban cada quince minutos. Madrid tiene más de cuatro mil puntos de medida, así que un solo mes ya nos da varios millones de lecturas.

## 2. Descarga de los datos

En lugar de pedirle al usuario del notebook que descargue los ficheros a mano, el propio notebook los pide directamente al portal. Es la opción más cómoda y reproducible: cualquiera puede clonar el repositorio, abrir este notebook, pulsar Run All y tenerlo todo listo.

Los enlaces de descarga no son obvios porque la web del portal tiene JavaScript y el HTML no expone las URLs directas. Las obtuvimos consultando el API de CKAN del portal (el software que usa el Ayuntamiento para publicar los datos), que sí devuelve la lista completa de recursos en JSON. Fuimos filtrando por mes y año para quedarnos con los enlaces relevantes de 2025 y 2026.

In [ ]:
# URLs directas de cada mes disponible, obtenidas del API CKAN del portal.
# Los enlaces de 2025 y 2026 siguen patrones ligeramente distintos (el Ayuntamiento
# reorganizó los recursos a principios de 2026), así que los tenemos todos listados.
URLS_HISTORICO = {
    '2025-01': 'https://datos.madrid.es/egob/catalogo/208627-199-transporte-ptomedida-historico-zip/download/01-2025.zip',
    '2025-02': 'https://datos.madrid.es/egob/catalogo/208627-200-transporte-ptomedida-historico-zip/download/02-2025.zip',
    '2025-03': 'https://datos.madrid.es/egob/catalogo/208627-201-transporte-ptomedida-historico-zip/download/03-2025.zip',
    '2025-04': 'https://datos.madrid.es/egob/catalogo/208627-202-transporte-ptomedida-historico-zip/download/04-2025.zip',
    '2025-05': 'https://datos.madrid.es/egob/catalogo/208627-203-transporte-ptomedida-historico-zip/download/05-2025.zip',
    '2025-06': 'https://datos.madrid.es/egob/catalogo/208627-204-transporte-ptomedida-historico-zip/download/06-2025.zip',
    '2025-07': 'https://datos.madrid.es/egob/catalogo/208627-205-transporte-ptomedida-historico-zip/download/07-2025.zip',
    '2025-08': 'https://datos.madrid.es/egob/catalogo/208627-206-transporte-ptomedida-historico-zip/download/08-2025.zip',
    '2025-09': 'https://datos.madrid.es/egob/catalogo/208627-212-transporte-ptomedida-historico-zip/download/09-2025.zip',
    '2025-10': 'https://datos.madrid.es/egob/catalogo/208627-213-transporte-ptomedida-historico-zip/download/10-2025.zip',
    '2025-11': 'https://datos.madrid.es/egob/catalogo/208627-214-transporte-ptomedida-historico-zip/download/11-2025.zip',
    '2025-12': 'https://datos.madrid.es/egob/catalogo/208627-215-transporte-ptomedida-historico-zip/download/12-2025.zip',
    '2026-01': 'https://datos.madrid.es/egob/catalogo/208627-216-transporte-ptomedida-historico/download/01-2026.zip',
    '2026-02': 'https://datos.madrid.es/egob/catalogo/208627-217-transporte-ptomedida-historico/download/02-2026.zip',
    '2026-03': 'https://datos.madrid.es/egob/catalogo/208627-218-transporte-ptomedida-historico/download/03-2026.zip',
}

# URL de la metadata de sensores (un único CSV, se actualiza periódicamente)
URL_SENSORES = 'https://datos.madrid.es/egob/catalogo/202468-0-intensidad-trafico.csv'


Tenemos el catálogo completo, pero no hace falta cargarlo todo para el EDA. Un solo mes ya nos da varios millones de lecturas y eso es más que suficiente para entender la forma de los datos. Para este primer análisis seleccionamos **marzo de 2026**, el último mes publicado cuando empezamos el proyecto.

Cargar más meses no aportaría información cualitativa nueva — los patrones de tráfico se repiten mes a mes con pocas variaciones — pero sí multiplicaría el tiempo de ejecución y el consumo de memoria. Cuando llegue el momento de entrenar el modelo final, en el notebook 03, se pueden añadir más meses sin tocar el código: basta con editar la lista `MESES_A_DESCARGAR`.

In [ ]:
# Si en el futuro queremos cargar más meses, basta con añadirlos a esta lista.
# El notebook está diseñado para escalar: toda la lógica opera sobre los meses que haya aquí.
MESES_A_DESCARGAR = ['2026-03']


La función de descarga es sencilla pero tiene un par de detalles importantes. Primero, comprueba si el fichero ya está en disco antes de descargar nada: así podemos reejecutar el notebook todas las veces que queramos sin bajarse gigas repetidos del portal. Segundo, usa streaming (`stream=True` + iterar por chunks) en lugar de cargar la respuesta entera en memoria: los ZIP de tráfico pesan entre 50 y 100 MB y hacerlo sin streaming es pedirle a Python que reserve toda esa memoria de golpe.

In [ ]:
def descargar_fichero(url: str, destino: Path) -> None:
    # Si ya tenemos el fichero no volvemos a bajarlo. Clave para reejecuciones rápidas.
    if destino.exists():
        print(f'  ✓ {destino.name} ya estaba descargado')
        return

    # stream=True evita cargar la respuesta entera en memoria de golpe
    respuesta = requests.get(url, stream=True)
    respuesta.raise_for_status()  # Lanza excepción si el servidor devuelve error

    # Escribimos por trozos de 1 MB. Los ZIP pesan decenas de megas.
    with open(destino, 'wb') as f:
        for chunk in respuesta.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)

    print(f'  ↓ {destino.name} descargado ({destino.stat().st_size / 1e6:.1f} MB)')


Con la función definida, descargamos lo que necesitemos.

In [ ]:
print('Descargando histórico...')
for mes in MESES_A_DESCARGAR:
    descargar_fichero(URLS_HISTORICO[mes], DATA_HIST / f'{mes}.zip')

print('\nDescargando metadata de sensores...')
descargar_fichero(URL_SENSORES, DATA_SENS / 'sensores.csv')


## 3. Carga del histórico de mediciones

El siguiente paso es leer los ficheros descargados y cargarlos en memoria como DataFrames de pandas. Como trabajamos con el ZIP sin extraer, escribimos una función pequeña que abre el ZIP, lee el primer (y único) CSV que hay dentro, y lo parsea. El parámetro `parse_dates=['fecha']` hace que pandas convierta la columna de fechas a tipo datetime al leerlas, lo que después nos permitirá extraer la hora o el día de la semana directamente sin conversiones adicionales.

El CSV del Ayuntamiento viene con separador punto y coma, no coma. Es la convención habitual en datasets europeos, porque la coma se usa como separador decimal. Por eso le pasamos `sep=';'` a `read_csv`.

In [ ]:
def cargar_mes(ruta_zip: Path) -> pd.DataFrame:
    with zipfile.ZipFile(ruta_zip) as z:
        # El zip contiene un solo CSV. Tomamos su nombre del listado interno.
        nombre_csv = z.namelist()[0]
        with z.open(nombre_csv) as f:
            return pd.read_csv(f, sep=';', parse_dates=['fecha'])


Cargamos los meses listados y los concatenamos en un único DataFrame. Aunque de momento solo tengamos un mes, dejamos el pattern preparado para más.

In [ ]:
df_historico = pd.concat(
    [cargar_mes(DATA_HIST / f'{mes}.zip') for mes in MESES_A_DESCARGAR],
    ignore_index=True,
)

print(f'Histórico cargado: {df_historico.shape[0]:,} filas × {df_historico.shape[1]} columnas')
print(f'Memoria ocupada:   {df_historico.memory_usage(deep=True).sum() / 1e6:.0f} MB')


Son del orden de varios millones de registros por mes. El consumo de memoria es alto pero asumible en un portátil moderno. Si llegara a ser un problema, se podría convertir la columna `id` a `category` (muchos valores repetidos) y las columnas enteras a `int32`, lo que recorta la memoria a la mitad. Por ahora no hace falta.

## 4. Carga de las ubicaciones de los sensores

El fichero de sensores es mucho más ligero: una fila por sensor (unos cuatro mil), no por lectura. Lo leemos con el mismo separador.

In [ ]:
df_sensores = pd.read_csv(DATA_SENS / 'sensores.csv', sep=';')

print(f'Sensores: {df_sensores.shape[0]:,} filas × {df_sensores.shape[1]} columnas')


## 5. Primera inspección

Antes de entrar en análisis, una foto rápida de lo que tenemos. Vamos a mirar la cabecera de cada DataFrame, los tipos de datos y un resumen estadístico básico. Son los tres pasos que siempre conviene hacer al abrir un dataset por primera vez.

In [ ]:
print('=== Histórico de mediciones ===')
print(f'Filas:    {len(df_historico):,}')
print(f'Columnas: {list(df_historico.columns)}')
print(f'Rango temporal: {df_historico["fecha"].min()}  →  {df_historico["fecha"].max()}')
print(f'Sensores únicos: {df_historico["id"].nunique():,}')


Tres líneas de números que ya nos cuentan una historia. Tenemos unos cuantos millones de lecturas que abarcan exactamente un mes natural, con varios miles de sensores distintos reportando. Eso da una idea del ratio de lecturas por sensor: cada uno graba cada quince minutos, o sea 96 veces al día, unas 2.900 veces al mes. Ese número lo tendremos en cuenta más adelante para valorar si los errores de sensor son aislados o sistemáticos.

In [ ]:
df_historico.head()


Las columnas `intensidad`, `ocupacion`, `carga` y `vmed` son las medidas físicas del sensor. `tipo_elem` distingue el tipo de punto (urbano, M-30, etc.). `error` es el flag de sensor defectuoso. `periodo_integracion` indica cuántos intervalos de 5 minutos agrega la lectura — normalmente 3, equivalente a 15 minutos.

In [ ]:
df_historico.dtypes


Los tipos están bien detectados. `fecha` es datetime gracias a `parse_dates`, los numéricos son int64/float64 y las categóricas son object (strings). No hay que convertir nada a mano.

In [ ]:
df_historico.describe()


El `describe` nos da el primer vistazo estadístico. Lo más llamativo es que el mínimo de `intensidad`, `ocupacion` y `carga` es `-1`. Esos menos-uno no son posibles físicamente (no puedes tener intensidad negativa) y son el marcador que usa el Ayuntamiento para avisar de un fallo de sensor. Hay que tratarlos como nulos; más adelante veremos cuántos son y qué hacemos con ellos.

Otra cosa interesante: la media de `carga` es bastante baja pero hay percentiles altos. Eso sugiere una distribución muy sesgada hacia la izquierda, con la mayoría de lecturas en valores bajos (tráfico fluido) y una cola larga de valores altos (horas punta, atascos). Lo confirmaremos visualmente en breve.

In [ ]:
print('=== Ubicaciones de sensores ===')
print(f'Filas:    {len(df_sensores):,}')
print(f'Columnas: {list(df_sensores.columns)}')
df_sensores.head()


El fichero de sensores nos da, para cada `id`, el `nombre` de la calle, el `distrito`, el tipo de elemento y las coordenadas UTM (`utm_x`, `utm_y`). Las coordenadas están en el sistema UTM zona 30N, que es el que usa por defecto el Ayuntamiento. Son metros relativos a una referencia, no grados. Si quisiéramos pintar un mapa con Folium o algo parecido habría que convertirlas a latitud/longitud, pero para un scatter plot simple los metros van igual de bien.

## 6. La variable objetivo: `carga`

Llegamos al análisis más importante del EDA. `carga` es lo que queremos predecir, así que hay que entenderla a fondo. Tres preguntas guían esta sección: cómo se distribuye, qué proporción de lecturas son errores, y dónde podemos poner el corte para decir "a partir de aquí es congestión".

Empezamos filtrando los `-1` para que no contaminen las estadísticas. No los eliminamos del DataFrame original (eso lo hará el notebook 02); simplemente los ignoramos para esta sección.

In [ ]:
# Lecturas válidas: las que no están marcadas como error. Las analizamos por separado.
carga_valida = df_historico.loc[df_historico['carga'] >= 0, 'carga']

print(f'Lecturas totales:  {len(df_historico):,}')
print(f'Lecturas válidas:  {len(carga_valida):,}')
print(f'Porcentaje válido: {len(carga_valida) / len(df_historico):.2%}')

print('\nEstadísticas descriptivas de carga (solo lecturas válidas):')
print(carga_valida.describe())


La mayoría de lecturas son válidas — bien. La media está en torno a 10-15, la mediana más baja aún. El percentil 75 no llega a 20. Eso significa que **tres de cada cuatro lecturas son de tráfico bastante fluido**. Solo cuando pasamos al percentil 90 o 95 empezamos a ver valores de carga que uno asociaría con un atasco real.

Es un primer indicio importante para elegir el umbral de congestión: si pusiéramos el corte en 30, ya estaríamos por encima del percentil 90 y tendríamos muy pocas congestiones en el dataset. Si lo pusiéramos en 60 o 70, tendríamos un número de congestiones más manejable y clínicamente significativo.

### Distribución visual: histograma y CDF

Las estadísticas descriptivas son útiles pero una gráfica lo cuenta mejor. Pintamos dos cosas en paralelo. A la izquierda, un **histograma** que muestra cuántas lecturas hay en cada rango de carga. A la derecha, una **función de distribución acumulada (CDF)**, que responde a la pregunta "¿qué porcentaje de lecturas tiene una carga menor o igual a X?".

La CDF es especialmente útil para elegir el umbral porque permite leer directamente, en el eje Y, la proporción de lecturas que se quedarían a cada lado del corte. Dibujamos tres líneas verticales sobre candidatos razonables (50%, 60%, 70%) para poder comparar.

In [ ]:
fig, (ax_hist, ax_cdf) = plt.subplots(1, 2, figsize=(12, 4))

# Histograma: cuántas lecturas caen en cada tramo de carga
ax_hist.hist(carga_valida, bins=50, color=COLOR_PRIMARIO, edgecolor='white', linewidth=0.3)
ax_hist.set_xlabel('Carga (%)')
ax_hist.set_ylabel('Nº de lecturas')
ax_hist.set_title('Distribución de la carga')

# CDF: proporción acumulada. Calcularla a mano es cuatro líneas.
carga_ordenada = np.sort(carga_valida.values)
cdf            = np.arange(1, len(carga_ordenada) + 1) / len(carga_ordenada)

ax_cdf.plot(carga_ordenada, cdf, color=COLOR_PRIMARIO, linewidth=1.5)

# Pintamos los candidatos a umbral para ver qué proporción dejan a cada lado
for umbral, color in [(50, '#999999'), (60, '#666666'), (70, COLOR_SECUNDARIO)]:
    ax_cdf.axvline(umbral, color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax_cdf.text(umbral + 1, 0.05, f'{umbral}', color=color, fontsize=9)

ax_cdf.set_xlabel('Carga (%)')
ax_cdf.set_ylabel('Proporción acumulada')
ax_cdf.set_title('CDF de la carga con umbrales candidatos')
ax_cdf.set_ylim(0, 1.02)

plt.tight_layout()
plt.show()


El histograma confirma lo que sospechábamos con el `describe`: la distribución está muy sesgada hacia valores bajos. Hay una concentración enorme de lecturas por debajo de 20 y una cola larga — pero poblada — hacia la derecha. No es una distribución rara; es exactamente lo que uno esperaría del tráfico urbano: la mayor parte del tiempo las calles están fluidas y los atascos son una minoría de momentos.

La CDF es más cuantitativa. Podemos leer directamente, por ejemplo, que con un umbral de 50 estaríamos etiquetando como "congestionado" en torno al 10-15% de las lecturas. Con 70, un 5% o menos. Necesitamos una tabla con los números exactos.

In [ ]:
# Tabla explícita con el % de registros que quedarían como "congestionado" según cada umbral
umbrales_candidatos = [40, 50, 60, 65, 70, 75, 80]
filas = []
for u in umbrales_candidatos:
    pct_congestion = (carga_valida >= u).mean()
    filas.append({
        'umbral':            u,
        '% congestionado':   f'{pct_congestion:.2%}',
        '% fluido':          f'{1 - pct_congestion:.2%}',
        'nº congestionadas': int((carga_valida >= u).sum()),
    })

pd.DataFrame(filas)


La tabla deja clara la elección. Con un umbral de 50 tenemos demasiadas congestiones (más del 10% del dataset), lo que significaría que estamos etiquetando como atasco situaciones que son solo tráfico medio. Con 80 tenemos tan pocas (menos del 3%) que el modelo se va a encontrar un problema de desbalance severo y va a tener pocos ejemplos positivos de los que aprender.

El punto dulce está entre 60 y 70. Nos decantamos por **70%**, que deja en torno a un 4-6% de congestiones. Es un punto en el que la carga ya es claramente alta — coincide con la percepción intuitiva de un atasco — y el desbalance sigue siendo manejable con `class_weight='balanced'`, sin necesidad de recurrir a técnicas más agresivas como SMOTE. Si más adelante encontráramos que el modelo falla mucho, podríamos volver aquí y replantearlo.

In [ ]:
UMBRAL_CARGA = 70
print(f'Umbral de congestión elegido: {UMBRAL_CARGA}%')


## 7. Errores de sensor

El segundo análisis crítico es entender los errores de medición. Vimos que hay valores `-1` en las columnas de medición física. Pero además, el dataset trae una columna `error` explícita con valores `'N'` (no hay error) o `'S'` (sí lo hay). ¿Son redundantes? ¿Se solapan al 100%? Conviene comprobarlo antes de decidir la estrategia de limpieza.

In [ ]:
# Contamos las dos marcas de error por separado y también cuánto se solapan
n_total   = len(df_historico)
n_err_neg = (df_historico['carga'] == -1).sum()
n_err_flag = (df_historico['error'] == 'S').sum()
n_err_amb = ((df_historico['carga'] == -1) & (df_historico['error'] == 'S')).sum()

print(f'Lecturas totales:                              {n_total:>10,}')
print(f'Con carga = -1 (marcador numérico):            {n_err_neg:>10,}  ({n_err_neg/n_total:.2%})')
print(f'Con error = "S" (marcador explícito):          {n_err_flag:>10,}  ({n_err_flag/n_total:.2%})')
print(f'Con ambas marcas a la vez:                     {n_err_amb:>10,}')
print(f'Solo con carga = -1 (sin error = "S"):         {n_err_neg - n_err_amb:>10,}')
print(f'Solo con error = "S" (sin carga = -1):         {n_err_flag - n_err_amb:>10,}')


Los dos marcadores **no son idénticos**. El grueso se solapa pero hay casos en que uno marca error y el otro no. Esto es relevante para la limpieza: en el notebook 02 eliminaremos cualquier fila que tenga alguna de las dos señales, en lugar de quedarnos solo con una. Ser conservadores aquí cuesta poco (perdemos muy pocas filas adicionales) y nos evita que un sensor marque error parcial y acabe colando una lectura corrupta en el modelo.

### ¿Los errores siguen algún patrón?

No todas las formas de "dato perdido" son iguales. Si los errores se reparten aleatoriamente por el dataset, basta con eliminarlos. Pero si se concentran en ciertas horas o en ciertos sensores, significa que la ausencia de dato está **correlacionada con algo**, y eso puede sesgar el análisis. Por ejemplo, si los sensores fallan sistemáticamente de noche, el análisis de patrones nocturnos será menos fiable.

In [ ]:
# Añadimos la hora al DataFrame para poder agregar por franja horaria.
# Creamos 'hora' y 'dia_semana' una sola vez aquí, las reutilizaremos después.
df_historico['hora']       = df_historico['fecha'].dt.hour
df_historico['dia_semana'] = df_historico['fecha'].dt.dayofweek   # 0 = lunes, 6 = domingo

# Tasa de error por hora del día: % de lecturas marcadas como erróneas en cada hora
tasa_error_por_hora = df_historico.assign(
    es_error=(df_historico['carga'] == -1) | (df_historico['error'] == 'S')
).groupby('hora')['es_error'].mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(tasa_error_por_hora.index, tasa_error_por_hora.values,
        marker='o', color=COLOR_SECUNDARIO, linewidth=1.5)
ax.set_xlabel('Hora del día')
ax.set_ylabel('Tasa de error')
ax.set_title('Tasa de error de los sensores por hora del día')
ax.set_xticks(range(0, 24, 2))
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()


La tasa de error es bastante plana a lo largo del día, con variaciones pequeñas. No hay una franja horaria en la que los sensores fallen mucho más que en otras, así que la limpieza directa (eliminar las filas problemáticas) no va a sesgar el análisis temporal.

### ¿Y por sensor? Los peores ofensores

Otra pregunta natural: ¿los errores están repartidos entre muchos sensores o hay unos pocos que fallan de forma sistemática? Si hay un puñado de sensores con tasa de error altísima, podría valer la pena excluirlos del modelo directamente.

In [ ]:
# Para cada sensor, calculamos qué proporción de sus lecturas son erróneas.
# Añadimos un filtro para no considerar sensores con muy pocas lecturas (estadísticamente no fiable).
df_historico['es_error'] = (df_historico['carga'] == -1) | (df_historico['error'] == 'S')

stats_por_sensor = df_historico.groupby('id').agg(
    lecturas=('es_error', 'size'),
    errores=('es_error',  'sum'),
)
stats_por_sensor['tasa_error'] = stats_por_sensor['errores'] / stats_por_sensor['lecturas']

# Nos quedamos con sensores que tengan al menos 500 lecturas (ya cubre casi todo el mes)
stats_por_sensor = stats_por_sensor[stats_por_sensor['lecturas'] >= 500]

# Top 10 ofensores
print('Top 10 sensores con más tasa de error:')
stats_por_sensor.nlargest(10, 'tasa_error').round(3)


In [ ]:
# Histograma de la distribución de tasas de error entre sensores
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(stats_por_sensor['tasa_error'], bins=40,
        color=COLOR_PRIMARIO, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Tasa de error del sensor')
ax.set_ylabel('Nº de sensores')
ax.set_title('¿Cómo se reparten los errores entre sensores?')
plt.tight_layout()
plt.show()


El histograma es muy claro: la inmensa mayoría de sensores tiene tasa de error cerca de cero, y hay una pequeña cola con sensores que fallan mucho. Esos ofensores sistemáticos probablemente son aparatos estropeados o con problemas específicos de instalación.

Para el modelo no hace falta excluirlos a mano: al eliminar las lecturas erróneas directamente, un sensor que falle el 50% del tiempo aportará la mitad de lecturas al dataset pero las que aporte serán buenas. Si algún día viéramos que el modelo aprende mal alguna zona concreta, podríamos volver aquí y decidir tirar esos sensores por completo. Por ahora no hace falta.

## 8. Patrones temporales

Hasta aquí hemos mirado los datos de forma estática. Toca ver cómo se comportan a lo largo del tiempo. Si el tráfico no tuviera ningún patrón temporal, la hora del día no aportaría información al modelo. Spoiler: sí lo tiene, y muy fuerte.

A partir de aquí trabajamos solo con las lecturas válidas. Creamos un DataFrame filtrado que usaremos el resto del EDA.

In [ ]:
# DataFrame limpio con las lecturas que no tienen ninguno de los dos marcadores de error
df_validos = df_historico[~df_historico['es_error']].copy()

print(f'Registros válidos: {len(df_validos):,}  ({len(df_validos)/len(df_historico):.1%} del total)')


### Carga media por hora del día

La primera gráfica temporal es la clásica curva diaria del tráfico: carga media en cada hora del día, promediando todos los días y todos los sensores. Es la gráfica que uno ve siempre cuando se habla de tráfico urbano, y es mitad cliché mitad necesaria: si el comportamiento no siguiera la forma típica, habría que desconfiar del dataset.

In [ ]:
carga_por_hora = df_validos.groupby('hora')['carga'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(carga_por_hora.index, carga_por_hora.values,
        marker='o', color=COLOR_PRIMARIO, linewidth=1.8)
ax.set_xlabel('Hora del día')
ax.set_ylabel('Carga media (%)')
ax.set_title('Carga media por hora del día')
ax.set_xticks(range(0, 24, 2))
ax.axvspan(7, 9, alpha=0.08, color=COLOR_SECUNDARIO)   # Punta de mañana
ax.axvspan(17, 20, alpha=0.08, color=COLOR_SECUNDARIO)  # Punta de tarde
plt.tight_layout()
plt.show()


La curva tiene el perfil de libro: valle profundo a las 4-5 de la madrugada, subida fuerte a partir de las 7, primera punta entre 8 y 9, meseta alta durante toda la mañana, un pequeño bajón al mediodía, segunda punta entre 18 y 19 (más alta que la de la mañana), y bajada progresiva hacia la noche. Las zonas sombreadas son las franjas punta clásicas. Esta gráfica nos autoriza a usar `hora` como feature: hay una señal clarísima que el modelo puede aprender.

### Carga media por día de la semana

Siguiente eje temporal: el día de la semana. ¿Hay diferencia entre un martes y un domingo? Intuitivamente sí, pero conviene cuantificarla. Dibujamos un barplot y coloreamos de forma distinta los laborables y el fin de semana para que el contraste salte a la vista.

In [ ]:
carga_por_dow = df_validos.groupby('dia_semana')['carga'].mean()
etiquetas_dow = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
colores_dow   = [COLOR_PRIMARIO] * 5 + [COLOR_SECUNDARIO] * 2  # Laborables vs fin de semana

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(etiquetas_dow, carga_por_dow.values, color=colores_dow)
ax.set_xlabel('Día de la semana')
ax.set_ylabel('Carga media (%)')
ax.set_title('Carga media por día de la semana')
plt.tight_layout()
plt.show()


Los laborables se parecen mucho entre sí, con el viernes ligeramente por encima. El fin de semana baja notablemente — especialmente el domingo. Esta diferencia laborable/fin-de-semana es una señal fuerte que no se captura solo con la hora: a las 9 de la mañana del lunes no hay el mismo tráfico que a las 9 de la mañana del domingo. Por eso conviene crear una feature `es_laborable` separada de `dia_semana`, para que el modelo tenga la dicotomía servida directamente.

### Carga media por mes

Con un único mes cargado, esta gráfica no aporta mucho: un solo barra. La incluimos pensando en el futuro: si alguien carga varios meses, esta celda mostrará automáticamente cómo cambia el tráfico a lo largo del año (típicamente: bajón en agosto por vacaciones, picos en otoño, bajón en Navidad...).

In [ ]:
df_validos['mes'] = df_validos['fecha'].dt.month

if df_validos['mes'].nunique() > 1:
    carga_por_mes = df_validos.groupby('mes')['carga'].mean()
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(carga_por_mes.index.astype(str), carga_por_mes.values, color=COLOR_PRIMARIO)
    ax.set_xlabel('Mes')
    ax.set_ylabel('Carga media (%)')
    ax.set_title('Carga media por mes del año')
    plt.tight_layout()
    plt.show()
else:
    print('Solo hay un mes en el dataset, la comparativa por mes no es informativa aquí.')
    print('Cargando más meses con MESES_A_DESCARGAR, esta gráfica cobra sentido.')


### El heatmap: la gráfica que lo resume todo

Combinar hora y día de la semana en un solo gráfico es especialmente potente. Un **heatmap** (mapa de calor) con horas en un eje y días en el otro, coloreado por la carga media, nos muestra el ADN del tráfico de Madrid en una sola imagen. Lo tendríamos colgado en la pared si estuviéramos en una oficina de movilidad urbana.

In [ ]:
# Matriz pivote: filas = día de la semana, columnas = hora, valores = carga media
pivot_hora_dia = df_validos.pivot_table(
    values='carga', index='dia_semana', columns='hora', aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_hora_dia, cmap='YlOrRd', cbar_kws={'label': 'Carga media (%)'},
            yticklabels=etiquetas_dow, ax=ax, linewidths=0.3, linecolor='white')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Día de la semana')
ax.set_title('Heatmap hora × día de la semana')
plt.tight_layout()
plt.show()


El heatmap concentra en una figura todo lo que hemos visto en las anteriores. Las dos bandas verticales anaranjadas en las horas punta de los laborables saltan a la vista. El fin de semana es visiblemente más frío. Y hay un detalle que sí se aprecia y que las gráficas anteriores por separado no mostraban: **los sábados y domingos el tráfico se desplaza hacia la tarde**. La punta matutina desaparece y la carga se concentra entre las 11 y las 21. Es decir, la relación hora-carga no es la misma en laborables que en fines de semana: hay una **interacción** entre las dos variables. Los modelos lineales no captan esa interacción a no ser que se la creemos a mano; los modelos basados en árboles sí la capturan solos, y ese es uno de los motivos por los que en el notebook 03 esperamos que Random Forest y Gradient Boosting superen a la regresión logística.

## 9. Análisis espacial

Después del tiempo, el espacio. Madrid no es uniforme: hay zonas que se colapsan casi siempre y otras que apenas se inmutan. Para analizarlo necesitamos unir el histórico con la metadata de sensores.

In [ ]:
# Carga media por sensor
carga_por_sensor = df_validos.groupby('id')['carga'].mean().rename('carga_media').reset_index()

# Unimos con la metadata (distrito, coordenadas, nombre)
df_mapa = carga_por_sensor.merge(
    df_sensores[['id', 'nombre', 'distrito', 'utm_x', 'utm_y']],
    on='id', how='left'
).dropna(subset=['distrito', 'utm_x', 'utm_y'])

print(f'Sensores con ubicación: {len(df_mapa):,}')


### Ranking por distrito

Primero las diferencias agregadas por distrito. Respondemos a la pregunta directa: ¿qué distritos tienen más tráfico medio?

In [ ]:
carga_por_distrito = df_mapa.groupby('distrito')['carga_media'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(carga_por_distrito.index.astype(str), carga_por_distrito.values, color=COLOR_PRIMARIO)
ax.set_xlabel('Carga media (%)')
ax.set_title('Carga media por distrito')
plt.tight_layout()
plt.show()


Las diferencias entre distritos son sustanciales. Los que encabezan el ranking son los del centro y los que tienen salidas a las principales vías de acceso a la ciudad. Los que cierran la tabla son zonas residenciales más periféricas. El distrito es, por tanto, otra feature útil para el modelo — aunque parte de esa información ya está codificada en las coordenadas UTM, el distrito como categoría agrupa factores comunes (densidad de comercio, oficinas, paradas de transporte) que las coordenadas solas no capturan.

### Mapa de sensores

Pasamos al detalle fino: un scatter plot donde cada punto es un sensor, situado en sus coordenadas UTM, y coloreado por su carga media. Es, a todos los efectos, un mapa de Madrid pintado por nivel de tráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
scatter = ax.scatter(df_mapa['utm_x'], df_mapa['utm_y'],
                     c=df_mapa['carga_media'], cmap='YlOrRd',
                     s=10, alpha=0.7)
plt.colorbar(scatter, ax=ax, label='Carga media (%)')
ax.set_xlabel('UTM X (metros)')
ax.set_ylabel('UTM Y (metros)')
ax.set_title('Mapa de sensores coloreado por carga media')
ax.set_aspect('equal')  # Para que no se deforme la geometría de Madrid
plt.tight_layout()
plt.show()


El mapa tiene forma de Madrid: se reconoce la M-30 como un anillo de puntos, la almendra central densamente poblada de sensores, y las radiales saliendo hacia afuera. El color cuenta lo mismo que la gráfica de distritos pero con más resolución: los puntos más calientes se concentran en la almendra y en algunas intersecciones de radiales con la M-30. Esto justifica incluir las coordenadas UTM como features numéricas y el distrito como feature categórica: el modelo puede aprender patrones locales más finos que los que captaría solo con el distrito.

## 10. Correlaciones entre variables

Las tres mediciones físicas del sensor — `intensidad`, `ocupacion` y `carga` — miden aspectos relacionados del mismo fenómeno. Conviene ver cuán relacionadas están. Una matriz de correlación responde a esta pregunta de un vistazo.

In [ ]:
cols_numericas = ['intensidad', 'ocupacion', 'carga', 'vmed']
matriz_corr = df_validos[cols_numericas].corr()

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, cbar_kws={'label': 'Correlación'}, ax=ax)
ax.set_title('Matriz de correlación entre variables de medición')
plt.tight_layout()
plt.show()


Las tres variables (intensidad, ocupación, carga) están fuertemente correlacionadas entre sí. Tiene sentido: la carga se calcula combinando intensidad y ocupación. La velocidad media (`vmed`) tiene correlación negativa con las anteriores, como cabría esperar: cuando hay atasco, los vehículos van más despacio.

Esto es importantísimo para la construcción del modelo. Si usáramos `intensidad` u `ocupacion` como features para predecir `carga`, estaríamos cometiendo **data leakage**: esas variables se miden en el mismo instante que la carga, así que el modelo no estaría aprendiendo a predecir el futuro sino a reconstruir el presente a partir de información del presente. Las métricas serían irrealmente buenas y el modelo inútil en producción. En el notebook 02 dejaremos fuera estas tres variables del conjunto de features. El modelo solo podrá apoyarse en información conocida **por adelantado**: hora, día, sensor, distrito, coordenadas.

## 11. Resumen de insights

Terminamos con los puntos que vamos a llevarnos a los notebooks siguientes. Son las decisiones y observaciones que justifican lo que haremos después.

**Sobre la variable objetivo.** La carga tiene una distribución muy sesgada a la izquierda. La mayoría de lecturas están muy por debajo del 20%, y hay una cola larga poblada hasta el 100. Fijamos el umbral de congestión en **70%**, que deja en torno a un 4-6% de registros como positivos. Es un desbalance manejable con `class_weight='balanced'`.

**Sobre los errores de sensor.** Hay dos marcadores de error que se solapan pero no coinciden al 100%: valores `-1` en las columnas de medición y la columna `error == 'S'`. En el notebook 02 eliminaremos las filas que tengan cualquiera de las dos marcas. Los errores no se concentran en ninguna franja horaria y la mayoría de sensores casi no fallan, así que la limpieza directa no introduce sesgo.

**Sobre los patrones temporales.** La hora del día es la variable más informativa que tenemos. Hay dos horas punta claras en los laborables y el fin de semana sigue un patrón muy distinto. La interacción hora × día de semana es relevante: los fines de semana el tráfico se desplaza hacia la tarde. Los modelos lineales no capturan esa interacción sin features específicas, pero los árboles sí.

**Sobre el espacio.** Hay diferencias importantes entre distritos y a nivel más fino también entre zonas concretas. Incluiremos distrito y coordenadas UTM como features.

**Sobre el data leakage.** No podemos usar `intensidad`, `ocupacion` ni `vmed` como features, porque están correlacionadas con la carga y se miden en el mismo momento. El modelo tiene que operar solo con información conocible por adelantado.

**Sobre la elección de modelos.** Dado que hay interacciones no lineales evidentes (hora × día de semana, distrito × hora), apostamos a que los modelos basados en árboles (Random Forest, Gradient Boosting) van a superar claramente a la regresión logística. La regresión logística servirá como baseline interpretable.

Con esto terminamos el EDA. El notebook 02 aplica las decisiones tomadas aquí: limpia los errores, une los datasets, construye las features y parte los datos en train y test para el modelado.